# HIBA metadata audit
Shared toolkit `skinaudit`; same checks as the other dataset audits.

Code: GPL-3.0-only, Copyright (C) 2026 Mohammad Hossein Omoumi (https://github.com/omoumi/skin-metadata-audit). The dataset keeps its own license.

In [ ]:
# install the toolkit: the attached Kaggle copy, otherwise the tagged GitHub release
import glob, os, shutil, subprocess, sys
REPO, TAG = "https://github.com/omoumi/skin-metadata-audit.git", "v0.1.0"
!pip install -q imagehash==4.3.2 scikit-learn==1.6.1 isic-cli==12.7.0
TOOLKIT = "/kaggle/temp/skin-metadata-audit"
local = sorted(glob.glob("/kaggle/input/**/skin-metadata-audit/pyproject.toml", recursive=True), key=len)
if local:
    shutil.copytree(os.path.dirname(local[0]), TOOLKIT, dirs_exist_ok=True)
elif not os.path.exists(os.path.join(TOOLKIT, "pyproject.toml")):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", TAG, REPO, TOOLKIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", TOOLKIT], check=True)
import skinaudit
print("skinaudit", skinaudit.__version__, "installed from", os.path.dirname(local[0]) if local else f"{REPO}@{TAG}")
# images go to /kaggle/temp, outside the saved notebook output
!test -f /kaggle/temp/hiba_raw/metadata.csv || isic image download --collections 175 /kaggle/temp/hiba_raw

In [ ]:
from skinaudit.adapters import hiba
OUT = "/kaggle/working/hiba_audit"
N_PERM = 200   # permutations of the null that gives the p-values
N_PERM_REF = 50   # permutations of the reference null (chance level of the gain, sensitivity)
N_PERM_MECH = "auto"   # label draws per missingness test: enough for Holm over the tested fields
ds = hiba.load("/kaggle/temp/hiba_raw")
print(ds.inputs); ds.df.head()

In [ ]:
import glob
from skinaudit.audit import run_audit
from skinaudit.dedup import load_confirmed
# confirmed duplicate pairs, from duplicates/ of the toolkit
dup_csv = os.path.join(TOOLKIT, "duplicates", f"{ds.spec.name}_confirmed_duplicates.csv")
dups = load_confirmed(dup_csv) if os.path.exists(dup_csv) else None   # no file: no confirmed duplicates
print("confirmed duplicate pairs:", 0 if dups is None else len(dups))
summary = run_audit(ds, OUT, n_perm=N_PERM, n_perm_reference=N_PERM_REF, n_boot=1000, duplicate_pairs=dups,
                     n_perm_mechanism=N_PERM_MECH)
summary["rubric"]

## Sample images for the README (only if the license allows it)

In [ ]:
from skinaudit.gallery import select_samples, make_gallery
gallery = None
if ds.spec.license.allows_redistribution:
    samples = select_samples(ds.df, ds.spec, n_per_class=3, prefer=ds.df["acquisition"].eq("dermoscopic"))
    gallery = make_gallery(samples, ds.spec, OUT, size=320)
gallery

## Near-duplicates (leakage between CV groups) and hashes for the cross-dataset check

In [ ]:
from skinaudit.dedup import phash_table, near_pairs, save_hashes
hashes = phash_table(ds.df)
save_hashes(hashes, f"{OUT}/tables/phash.csv")
pairs = near_pairs(hashes, max_dist=4)
pairs.to_csv(f"{OUT}/tables/near_duplicates.csv", index=False)
# pairs across CV groups: identical files, likely same photo, the rest for review by eye
from skinaudit.dedup import describe_pairs, identical_pairs, likely_same_photo, review_figure
os.makedirs(f"{OUT}/review", exist_ok=True)
cross = describe_pairs(pairs[pairs["crosses_groups"]], ds.df)
cross.to_csv(f"{OUT}/review/cross_group_pairs.csv", index=False)
same = identical_pairs(cross)
same.to_csv(f"{OUT}/review/identical_files.csv", index=False)
likely = likely_same_photo(cross)   # same photo saved again: pixel correlation >= 0.97
likely.to_csv(f"{OUT}/review/likely_same_photo.csv", index=False)
print(len(likely), "pairs are likely the same photo (check them first in the review figures)")
print(len(same), "identical files,", int((cross["identical_file"] & cross["labels_differ"]).sum()),
      "of them with different labels")
look = cross[~cross["identical_file"]].reset_index(drop=True)
look.to_csv(f"{OUT}/review/near_duplicates_to_review.csv", index_label="pair")
MAX_REVIEW = 300   # most similar pairs first
print(review_figure(look, ds.df, f"{OUT}/review/near_duplicates.jpg", max_pairs=MAX_REVIEW))
print(len(look), "pairs to check by eye;", min(len(look), MAX_REVIEW), "shown in the review figures")
print(len(pairs), "near-duplicate pairs;", int(pairs["crosses_groups"].sum()), "of them cross CV groups")
pairs.head(20)

In [ ]:
from skinaudit.report import write_readme
write_readme(OUT, gallery=gallery, reproduce="The notebook that produced these files is in [`notebooks/`](notebooks).")
print(open(f"{OUT}/README.md").read()[:2000])

## Dermoscopic-only audit

In [ ]:
ds_d = hiba.load("/kaggle/temp/hiba_raw", dermoscopic_only=True)
run_audit(ds_d, "/kaggle/working/hiba_audit/dermoscopic_only", n_perm=N_PERM, n_perm_reference=N_PERM_REF, duplicate_pairs=dups,
          n_perm_mechanism=N_PERM_MECH)["rubric"]

In [ ]:
# Zip the output folder for download
import shutil
print(shutil.make_archive(OUT, "zip", os.path.dirname(OUT), os.path.basename(OUT)))